In [19]:
from references import zeng24_config, zeng24_question
from src.retrieval import VectorRetriever, RerankerManager
from src.prompts import LLMQueryRewriter
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 初始化设置以及数据库

In [8]:
cfg = zeng24_config.Zeng24fiqa()

In [10]:
cfg

Zeng24fiqa(datastorage=vDataStorageConfig(data_name='fiqa', raw_data_dir=['./data/fiqa'], tool='vector-chroma'), chunk=vChunkConfig(params={}), embedding=vEmbeddingConfig(provider='hf', model_name='bge-large-en-v1.5', model_dir='./Models/BAAI-bge-large-en-v1.5'), llm=vLLMConfig(provider='hf', model_name='./Models/Qwen2.5-14B-Instruct', reasoning=True, vllm_parallel_size=2, vllm_gpu_memory_utilization=0.9, temperature=0, top_p=1, max_seq_len=4096, max_gen_len=4096), prompt=vPromptConfig(suffix=['context: ', 'question: ', 'answer:'], adhesive='\n'), retrieval=vRetrievalConfig(method='similarity_score_threshold', rerank='BAAI/bge-reranker-large', adhesive='\n\n', params={'k': 15, 'n': 10, 'score_threshold': 0.5}))

In [11]:
# 初始化
retriever = VectorRetriever(cfg, device='cpu', force_rebuild=False)

[INFO] Retrieval name: ./data/fiqa Store path: ./retrieval_stores/./data/fiqa/bge-large-en-v1.5/vector-chroma
[INFO] Loading existing Chroma DB: ./data/fiqa
Retriever of similarity_score_threshold is ready.
Retriever of vector-chroma is ready.
[INFO] Retriever for ./data/fiqa is ready!


# 生成或加载问题

In [33]:
# 输入查询
queries = ["Tell me about APPLE.", "Tell me about Google."]

In [37]:
qrw = LLMQueryRewriter(model="./Models/Qwen2.5-14B-Instruct", base_url="http://g56:22999/v1", api_key="EMPTY")

In [38]:
queries_rws = qrw.rewrite(queries, n_variants=5)

In [39]:
queries_rws

{'original_query': ['Tell me about APPLE.', 'Tell me about Google.'],
 'rewritten_queries': [['What can you tell me about the history of Apple Inc?',
   'Provide information on the products manufactured by Apple.',
   "What are some criticisms of Apple's business practices?",
   'How does Apple compare to other technology companies in terms of innovation?',
   'What are the benefits and drawbacks of using Apple devices?'],
  ['What is the history of Google?',
   'How did Google start and grow into what it is today?',
   "What are some criticisms of Google's practices?",
   'How does Google compare to other search engines in terms of user privacy?',
   'What are the benefits and drawbacks of using Google services?']],
 'all_queries': [['Tell me about APPLE.',
   'What can you tell me about the history of Apple Inc?',
   'Provide information on the products manufactured by Apple.',
   "What are some criticisms of Apple's business practices?",
   'How does Apple compare to other technolog

# 检索得到chunk

In [42]:
reranker = RerankerManager(cfg, device='cpu')

[INFO] Reranker BAAI/bge-reranker-large is ready!


In [44]:
# 调用 retrieve 方法
contexts, doc_ids = retriever.retrieve(queries_rws["original_query"])

# 查看结果
for i, q in enumerate(queries_rws["original_query"]):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: Tell me about APPLE.
  1. [137360] I can't decide what to do about apple. Huge market share, huge gobs of cash that they don't know wha...
  2. [358413] Apple's specialty is UX.  It's an incredibly talented UX company, both software and hardware wise.  ...
  3. [162405] I hate apples business practice down to the core, but whenever a not-so-tech-savvy friend or older g...
  4. [515651] The benefits Apple offers are pretty amazing. 15% discount on stock (you can allocate up to 10% of y...
  5. [488869] Apple is eating the lunch of Nokia, Rimm, and Sprint. A quick check of their balance sheets and fina...
  6. [115991] "What does your comment have to do with my comment? You say ""Apple only designs stuff"" as if that ...
  7. [470984] Seems pretty nice. My only real complaint with my current MacBook Pro is its heft. I opted for the s...
  8. [28862] "I know this I irrelevant, but whale. Man, what an og username.   I agree with you on the offshore b...
  9. [481136] um, no.   In

In [45]:
contexts, doc_ids  = reranker.rerank(contexts, doc_ids, queries)

# 查看结果
for i, q in enumerate(queries):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: Tell me about APPLE.
  1. [358413] Apple's specialty is UX.  It's an incredibly talented UX company, both software and hardware wise.  ...
  2. [137360] I can't decide what to do about apple. Huge market share, huge gobs of cash that they don't know wha...
  3. [343855] I wouldn't even say it's amazing at UX. Once upon a time it was, but these days it has lost a ton of...
  4. [28862] "I know this I irrelevant, but whale. Man, what an og username.   I agree with you on the offshore b...
  5. [162405] I hate apples business practice down to the core, but whenever a not-so-tech-savvy friend or older g...
  6. [515651] The benefits Apple offers are pretty amazing. 15% discount on stock (you can allocate up to 10% of y...
  7. [224808] Which in the spirit of the conversation, is pretty much true. You can argue with someone else about ...
  8. [115991] "What does your comment have to do with my comment? You say ""Apple only designs stuff"" as if that ...
  9. [488869] Apple is eat

#### 下面测试使用rewriter的格式

In [47]:
# 调用 retrieve 方法
contexts, doc_ids = retriever.retrieve(queries_rws["all_queries"])

# 查看结果
for i, q in enumerate(queries_rws["all_queries"]):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: ['Tell me about APPLE.', 'What can you tell me about the history of Apple Inc?', 'Provide information on the products manufactured by Apple.', "What are some criticisms of Apple's business practices?", 'How does Apple compare to other technology companies in terms of innovation?', 'What are the benefits and drawbacks of using Apple devices?']
  1. [137360] I can't decide what to do about apple. Huge market share, huge gobs of cash that they don't know wha...
  2. [358413] Apple's specialty is UX.  It's an incredibly talented UX company, both software and hardware wise.  ...
  3. [162405] I hate apples business practice down to the core, but whenever a not-so-tech-savvy friend or older g...
  4. [515651] The benefits Apple offers are pretty amazing. 15% discount on stock (you can allocate up to 10% of y...
  5. [488869] Apple is eating the lunch of Nokia, Rimm, and Sprint. A quick check of their balance sheets and fina...
  6. [115991] "What does your comment have to do with m

In [48]:
contexts, doc_ids  = reranker.rerank(contexts, doc_ids, queries)

# 查看结果
for i, q in enumerate(queries):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: Tell me about APPLE.
  1. [358413] Apple's specialty is UX.  It's an incredibly talented UX company, both software and hardware wise.  ...
  2. [137360] I can't decide what to do about apple. Huge market share, huge gobs of cash that they don't know wha...
  3. [229950] "The story with most companies, Tesla, Apple, Microsoft, old GM, etc is that innovation was done by ...
  4. [343855] I wouldn't even say it's amazing at UX. Once upon a time it was, but these days it has lost a ton of...
  5. [297512] I mean, they're actually pretty successful in their niche.  And Apple creates a huge positive extern...
  6. [28862] "I know this I irrelevant, but whale. Man, what an og username.   I agree with you on the offshore b...
  7. [440346] I really like wireless charging, not a deal breaker though. I find the larger screen more usable. I ...
  8. [172864] For Cook, it was finding Apple, a company that stood for something bigger than selling electronics a...
  9. [162405] I hate apple

# 形成结构prompt

In [49]:
from src.llm import OpenAILLM
from src.prompts import SimplePromptConstructor

In [51]:
p_construct = SimplePromptConstructor()

In [52]:
p_construct.prefix

['context: ', 'question: ', 'answer:']

In [53]:
end_ppt = p_construct.batch_construct(queries, contexts)

In [62]:
LL_Model = OpenAILLM(cfg, model="./Models/Qwen2.5-14B-Instruct", base_url="http://g56:22999/v1", api_key="EMPTY")

In [63]:
LL_Model.infer("who are you?")

'I am Qwen, a large language model created by Alibaba Cloud. I am designed to be helpful, honest, and harmless, and I aim to assist with a wide range of tasks and questions you might have. How can I assist you today?'

In [64]:
LL_Model.batch_infer(end_ppt)

["Apple Inc., often simply referred to as Apple, is a multinational technology company headquartered in Cupertino, California. Founded in 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne, Apple has grown from a small startup to one of the world's largest and most valuable companies. The company is renowned for its innovative approach to product design, user experience, and branding, and is known for a range of consumer electronics, computer software, and online services.\n\n### Key Products and Services\n\n- **iPhones**: Apple's flagship product, iPhones are smartphones known for their sleek design, user-friendly interface, and integration with other Apple products.\n- **Mac Computers**: Desktops and laptops designed for professional and personal use, known for their reliability and performance.\n- **iPad Tablets**: Portable tablets that offer a balance between the functionality of a computer and the convenience of a smartphone.\n- **Apple Watch**: A smartwatch that integrates seaml

In [65]:
LL_Model.batch_infer(queries)

['When you mention "APPLE," it\'s likely you\'re referring to Apple Inc., one of the world’s leading technology companies. Founded in 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne, Apple is headquartered in Cupertino, California. The company is renowned for its innovative consumer electronics, software, and services.\n\nSome of Apple\'s most famous products include:\n\n- **iMac**: A line of all-in-one desktop computers.\n- **MacBook**: A series of laptop computers.\n- **iPhone**: A line of smartphones that revolutionized the mobile phone industry.\n- **iPad**: A tablet computer designed for media consumption and light productivity tasks.\n- **Apple Watch**: A smartwatch that integrates health and fitness tracking with smartphone functionality.\n- **AirPods**: Wireless earbuds that offer seamless integration with other Apple devices.\n\nApple also develops and maintains several operating systems and software platforms:\n- **macOS**: For Mac computers.\n- **iOS**: For iPhones and i